In [ ]:
#!/usr/bin/env python3
"""
Lab MQTT — 02. Multi-broker MQTT collector
===========================================

Connects in parallel to several public MQTT brokers, subscribes (QoS 2, as
required) to the topic list produced by `01_topic_analysis.ipynb`, and logs
every received message to a gzip-compressed pickle, using the same schema as
the "MQTT in the Wild" paper:

    {
        "timestamp"      : float (epoch seconds),
        "broker"         : str,
        "topic"          : str,
        "topic_depth"    : int,   # number of levels
        "topic_length"   : int,   # bytes
        "qos"            : int    (0/1/2 — publisher original QoS),
        "retain"         : bool,
        "dup"            : bool,
        "payload_length" : int,
        "payload_type"   : str    ("json" | "string" | "numeric" | "bool" | "unknown"),
        "payload_head"   : bytes  (first 20 payload bytes, for later debug/decoding),
    }

Usage (CLI / local):
    python 02_mqtt_collector.py \\
        --subs ./outputs/subscription_list.json \\
        --duration 3600 \\
        --out ./outputs/captured_messages.pkl.gz

Usage (Kaggle notebook):
    # 1) Settings -> Internet: ON (needed for the public brokers).
    # 2) Upload as a Kaggle dataset the folder containing this file +
    #    subscription_list.json (or paste them into the working dir).
    # 3) In a cell:
    !pip -q install paho-mqtt

    # 4) "02_mqtt_collector" starts with a digit and is not a valid Python
    #    identifier: load the file as a module via importlib.util.
    import importlib.util, sys
    spec = importlib.util.spec_from_file_location(
        "mqtt_collector",
        "/kaggle/input/<your-dataset>/02_mqtt_collector.py",
    )
    mqtt_collector = importlib.util.module_from_spec(spec)
    sys.modules["mqtt_collector"] = mqtt_collector
    spec.loader.exec_module(mqtt_collector)

    # 5) Start the capture (output under /kaggle/working/ is downloadable after the run).
    mqtt_collector.run(
        subs_path="/kaggle/input/<your-dataset>/subscription_list.json",
        out_path="/kaggle/working/captured_messages.pkl.gz",
        duration=3600,
    )

    # Quick & dirty alternative: run the script as a separate process.
    #   !python /kaggle/input/<your-dataset>/02_mqtt_collector.py \
    #       --subs /kaggle/input/<your-dataset>/subscription_list.json \
    #       --out  /kaggle/working/captured_messages.pkl.gz \
    #       --duration 3600

Ctrl-C stops cleanly (all buffered messages are saved).

Dependencies:
    pip install paho-mqtt pandas
"""
from __future__ import annotations

import argparse
import gzip
import json
import os
import pickle
import signal
import sys
import threading
import time
from dataclasses import dataclass
from pathlib import Path
from queue import Queue, Empty
from typing import List, Dict, Any, Optional

try:
    import paho.mqtt.client as mqtt
except ImportError:
    # Auto-install on clean envs like Kaggle / Colab: the module is not
    # preinstalled, but pip is available.
    import subprocess
    print("[bootstrap] paho-mqtt not found, installing it with pip...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "paho-mqtt"]
    )
    import paho.mqtt.client as mqtt  # noqa: E402


# ---------------------------------------------------------------------------
# Kaggle detection + path defaults
# ---------------------------------------------------------------------------
IS_KAGGLE = bool(
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    or os.environ.get("KAGGLE_URL_BASE")
    or Path("/kaggle/working").exists()
)

if IS_KAGGLE:
    # On Kaggle inputs are mounted under /kaggle/input/<dataset-slug>/.
    # The dataset name is not known in advance: use the first subs file
    # found under /kaggle/input, or a placeholder the user can override
    # by passing subs_path to run().
    def _autodetect_kaggle_subs() -> Path:
        root = Path("/kaggle/input")
        if root.exists():
            for p in root.glob("*/subscription_list.json"):
                return p
            for p in root.rglob("subscription_list.json"):
                return p
        return root / "subscription_list.json"

    DEFAULT_SUBS_PATH = _autodetect_kaggle_subs()
    DEFAULT_OUT_PATH = Path("/kaggle/working/captured_messages.pkl.gz")
else:
    DEFAULT_SUBS_PATH = Path("./outputs/subscription_list.json")
    DEFAULT_OUT_PATH = Path("./outputs/captured_messages.pkl.gz")


# ---------------------------------------------------------------------------
# paho-mqtt v1/v2 compatibility shim
# ---------------------------------------------------------------------------
# paho-mqtt 2.x requires callback_api_version. The callbacks in this file
# use the legacy (V1) signature, supported by both versions as long as it
# is requested explicitly on 2.x.
def _make_mqtt_client(client_id: str) -> mqtt.Client:
    try:
        api_v1 = mqtt.CallbackAPIVersion.VERSION1  # paho 2.x
        return mqtt.Client(
            callback_api_version=api_v1,
            client_id=client_id,
            clean_session=True,
        )
    except AttributeError:
        # paho 1.x: no callback_api_version
        return mqtt.Client(client_id=client_id, clean_session=True)


# ---------------------------------------------------------------------------
# Default broker list (override with --brokers)
# ---------------------------------------------------------------------------
DEFAULT_BROKERS = [
    {"name": "mosquitto", "host": "test.mosquitto.org", "port": 1883},
    {"name": "hivemq",    "host": "broker.hivemq.com",  "port": 1883},
    {"name": "emqx",      "host": "broker.emqx.io",     "port": 1883},
    {"name": "emqx_cn",  "host": "broker-cn.emqx.io",     "port": 1883},
]


# ---------------------------------------------------------------------------
# Payload classification (same schema as the paper)
# ---------------------------------------------------------------------------
def classify_payload(payload: bytes) -> str:
    """Return 'json' | 'string' | 'numeric' | 'bool' | 'unknown'."""
    if not payload:
        return "unknown"
    # Try UTF-8 decoding (MQTT does not require it but it almost always holds)
    try:
        text = payload.decode("utf-8")
    except UnicodeDecodeError:
        return "unknown"
    text_strip = text.strip()
    # bool
    if text_strip.lower() in ("true", "false", "0", "1"):
        # 0/1 stays numeric, "true"/"false" becomes bool
        if text_strip.lower() in ("true", "false"):
            return "bool"
    # numeric
    try:
        float(text_strip)
        return "numeric"
    except ValueError:
        pass
    # json
    if text_strip and text_strip[0] in "{[":
        try:
            json.loads(text_strip)
            return "json"
        except json.JSONDecodeError:
            pass
    # fallback: printable string
    if all(32 <= b < 127 or b in (9, 10, 13) for b in payload):
        return "string"
    return "unknown"


# ---------------------------------------------------------------------------
# Per-broker worker
# ---------------------------------------------------------------------------
@dataclass
class BrokerCfg:
    name: str
    host: str
    port: int = 1883
    keepalive: int = 60


class BrokerWorker(threading.Thread):
    """One thread per broker. Puts each received message into the shared queue."""

    def __init__(self, cfg: BrokerCfg, subs: List[str], qos: int, queue: Queue,
                 stop_event: threading.Event):
        super().__init__(daemon=True, name=f"broker-{cfg.name}")
        self.cfg = cfg
        self.subs = subs
        self.qos = qos
        self.queue = queue
        self.stop_event = stop_event
        self.n_received = 0
        self.client = _make_mqtt_client(
            client_id=f"polimi-labexp-{cfg.name}-{int(time.time())}"
        )
        self.client.on_connect    = self._on_connect
        self.client.on_message    = self._on_message
        self.client.on_disconnect = self._on_disconnect

    # --- callbacks -----------------------------------------------------------
    def _on_connect(self, client, userdata, flags, rc):
        if rc == 0:
            print(f"[{self.cfg.name}] CONNECT OK")
            # Subscribe in batches
            sub_list = [(t, self.qos) for t in self.subs]
            # paho accepts lists of ~ a few hundred per call
            CHUNK = 100
            for i in range(0, len(sub_list), CHUNK):
                client.subscribe(sub_list[i:i+CHUNK])
            print(f"[{self.cfg.name}] subscribed to {len(self.subs)} filters @ QoS {self.qos}")
        else:
            print(f"[{self.cfg.name}] CONNECT FAILED, rc={rc}")

    def _on_disconnect(self, client, userdata, rc):
        if rc != 0:
            print(f"[{self.cfg.name}] unexpected disconnect (rc={rc}), will retry reconnect")

    def _on_message(self, client, userdata, msg):
        try:
            payload = bytes(msg.payload)
            record = {
                "timestamp"     : time.time(),
                "broker"        : self.cfg.name,
                "topic"         : msg.topic,
                "topic_depth"   : sum(1 for p in msg.topic.split('/') if p),
                "topic_length"  : len(msg.topic.encode("utf-8")),
                "qos"           : msg.qos,
                "retain"        : bool(msg.retain),
                "dup"           : bool(msg.dup),
                "payload_length": len(payload),
                "payload_type"  : classify_payload(payload),
                "payload_head"  : payload[:20],
            }
            self.queue.put(record)
            self.n_received += 1
        except Exception as e:
            print(f"[{self.cfg.name}] on_message error: {e}")

    # --- run loop ------------------------------------------------------------
    def run(self):
        backoff = 2
        while not self.stop_event.is_set():
            try:
                self.client.connect(self.cfg.host, self.cfg.port, self.cfg.keepalive)
                # loop_start spawns an internal thread; we wait until stop
                self.client.loop_start()
                while not self.stop_event.is_set():
                    self.stop_event.wait(timeout=1.0)
                self.client.loop_stop()
                try:
                    self.client.disconnect()
                except Exception:
                    pass
                break
            except Exception as e:
                print(f"[{self.cfg.name}] connection error: {e}, retry in {backoff}s")
                if self.stop_event.wait(timeout=backoff):
                    break
                backoff = min(backoff * 2, 60)


# ---------------------------------------------------------------------------
# Writer thread: periodically writes to disk
# ---------------------------------------------------------------------------
class Writer(threading.Thread):
    def __init__(self, queue: Queue, out_path: Path, stop_event: threading.Event,
                 flush_every: int = 5000, flush_seconds: float = 30.0):
        super().__init__(daemon=True, name="writer")
        self.queue = queue
        self.out_path = out_path
        self.stop_event = stop_event
        self.flush_every = flush_every
        self.flush_seconds = flush_seconds
        self.buffer: List[Dict[str, Any]] = []
        self.total_written = 0
        self.lock = threading.Lock()

    def _flush(self):
        if not self.buffer:
            return
        # Append-style: read existing, concatenate, rewrite.
        # For large datasets a record format (jsonl/parquet) would be better,
        # but for a 1-4h run this is fine.
        existing: List[Dict[str, Any]] = []
        if self.out_path.exists():
            try:
                with gzip.open(self.out_path, "rb") as fh:
                    existing = pickle.load(fh)
            except Exception as e:
                print(f"[writer] WARN: could not re-read {self.out_path}: {e}")
        existing.extend(self.buffer)
        tmp = self.out_path.with_suffix(self.out_path.suffix + ".tmp")
        with gzip.open(tmp, "wb") as fh:
            pickle.dump(existing, fh, protocol=pickle.HIGHEST_PROTOCOL)
        tmp.replace(self.out_path)
        self.total_written += len(self.buffer)
        print(f"[writer] flush: +{len(self.buffer)} messages (total on disk: {self.total_written})")
        self.buffer.clear()

    def run(self):
        last_flush = time.time()
        while not self.stop_event.is_set():
            try:
                rec = self.queue.get(timeout=1.0)
                self.buffer.append(rec)
            except Empty:
                pass
            now = time.time()
            if (len(self.buffer) >= self.flush_every or
                (self.buffer and now - last_flush >= self.flush_seconds)):
                with self.lock:
                    self._flush()
                last_flush = now
        # Final drain
        while True:
            try:
                self.buffer.append(self.queue.get_nowait())
            except Empty:
                break
        with self.lock:
            self._flush()


# ---------------------------------------------------------------------------
# Core run() — usable both from CLI and from a notebook (Kaggle / Jupyter)
# ---------------------------------------------------------------------------
def run(
    subs_path: Optional[os.PathLike] = None,
    out_path: Optional[os.PathLike] = None,
    duration: int = 3600,
    brokers: Optional[List[Dict[str, Any]]] = None,
    brokers_path: Optional[os.PathLike] = None,
) -> Dict[str, Any]:
    """Start the multi-broker capture.

    Args:
        subs_path:    JSON with schema {"qos": int, "topics": [...]}. Default: DEFAULT_SUBS_PATH.
        out_path:     Output .pkl.gz file. Default: DEFAULT_OUT_PATH.
        duration:     Capture seconds (0 = forever).
        brokers:      List of dicts {"name","host","port"} (direct override).
        brokers_path: JSON with a custom broker list (alternative to `brokers`).

    Returns:
        Dict with final stats (handy for Kaggle notebooks).
    """
    subs_path = Path(subs_path) if subs_path else DEFAULT_SUBS_PATH
    out_path = Path(out_path) if out_path else DEFAULT_OUT_PATH

    if not subs_path.exists():
        raise FileNotFoundError(
            f"subs_path not found: {subs_path}. "
            f"On Kaggle, upload subscription_list.json as a dataset "
            f"and pass the full path to run(subs_path=...)."
        )

    sub_cfg = json.loads(subs_path.read_text())
    topics: List[str] = sub_cfg["topics"]
    qos: int = int(sub_cfg.get("qos", 2))
    print(f"Loaded {len(topics)} subscribe filters @ QoS {qos} from {subs_path}")

    if brokers is not None:
        broker_dicts = brokers
    elif brokers_path is not None:
        broker_dicts = json.loads(Path(brokers_path).read_text())
    else:
        broker_dicts = DEFAULT_BROKERS

    out_path.parent.mkdir(parents=True, exist_ok=True)
    queue: Queue = Queue(maxsize=200_000)
    stop_event = threading.Event()

    workers = [
        BrokerWorker(BrokerCfg(**b), topics, qos, queue, stop_event)
        for b in broker_dicts
    ]
    writer = Writer(queue, out_path, stop_event)

    # Graceful shutdown — signal.signal only works in the process main thread.
    # On Kaggle/Jupyter the cell runs in the kernel main thread, so usually OK;
    # but if we are in a secondary thread we avoid a ValueError killing startup.
    def _handle_sig(*_):
        print("\n>>> Interrupt received, closing brokers and flushing to disk...")
        stop_event.set()

    try:
        signal.signal(signal.SIGINT, _handle_sig)
        signal.signal(signal.SIGTERM, _handle_sig)
    except (ValueError, OSError) as e:
        print(f"[main] signal handler not registered ({e}); "
              f"use stop_event manually if needed.")

    writer.start()
    for w in workers:
        w.start()

    deadline = (time.time() + duration) if duration > 0 else None
    try:
        while not stop_event.is_set():
            time.sleep(5)
            stats = ", ".join(f"{w.cfg.name}={w.n_received}" for w in workers)
            print(f"[main] queue={queue.qsize():>5}  received: {stats}")
            if deadline and time.time() >= deadline:
                stop_event.set()
    except KeyboardInterrupt:
        # On Kaggle/Jupyter the user presses "interrupt kernel": we catch it
        # here to avoid a stack trace and proceed with an orderly shutdown.
        print("\n>>> KeyboardInterrupt: starting orderly shutdown...")
        stop_event.set()
    finally:
        stop_event.set()
        for w in workers:
            w.join(timeout=10)
        writer.join(timeout=30)

    print("\n=== Final summary ===")
    per_broker = {}
    for w in workers:
        print(f"  {w.cfg.name:12s}  received = {w.n_received}")
        per_broker[w.cfg.name] = w.n_received
    print(f"  Total written to disk: {writer.total_written}")
    print(f"  Output file: {out_path}")

    return {
        "out_path": str(out_path),
        "total_written": writer.total_written,
        "per_broker_received": per_broker,
    }


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description="Multi-broker MQTT collector (QoS 2)")
    ap.add_argument("--subs", type=Path, default=DEFAULT_SUBS_PATH,
                    help=f"JSON file with schema {{'qos': 2, 'topics': [...]}} "
                         f"(default: {DEFAULT_SUBS_PATH})")
    ap.add_argument("--out", type=Path, default=DEFAULT_OUT_PATH,
                    help=f"Output file (.pkl.gz) (default: {DEFAULT_OUT_PATH})")
    ap.add_argument("--duration", type=int, default=3600,
                    help="Capture duration in seconds (default 3600 = 1h). 0 = forever.")
    ap.add_argument("--brokers", type=Path, default=None,
                    help="JSON with a custom broker list (default: mosquitto/hivemq/emqx)")
    args = ap.parse_args()

    run(
        subs_path=args.subs,
        out_path=args.out,
        duration=args.duration,
        brokers_path=args.brokers,
    )


def _in_jupyter_kernel() -> bool:
    """True if we are inside a Jupyter/Colab/Kaggle kernel (so sys.argv does
    NOT contain our CLI args, but the kernel launcher ones)."""
    if "ipykernel" in sys.modules or "google.colab" in sys.modules:
        return True
    argv0 = (sys.argv[0] or "").lower()
    if "ipykernel" in argv0 or "colab_kernel_launcher" in argv0:
        return True
    # typical: -f /path/kernel-xxx.json among the args
    if any(a == "-f" for a in sys.argv[1:]):
        return True
    return False


if __name__ == "__main__":
    if _in_jupyter_kernel():
        if IS_KAGGLE:
            # Kaggle Save & Commit runs the .py top-to-bottom like a notebook:
            # here we start the capture directly, using the Kaggle-aware
            # defaults (subscription_list.json autodetected under
            # /kaggle/input/*/, output in /kaggle/working/).
            # Duration is overridable via the MQTT_DURATION env var.
            _dur = int(os.environ.get("MQTT_DURATION", "172800"))
            print(f"[main] Kaggle kernel detected: starting run() (duration={_dur}s).")
            print(f"       subs_path = {DEFAULT_SUBS_PATH}")
            print(f"       out_path  = {DEFAULT_OUT_PATH}")
            run(duration=_dur)
        else:
            # %run from local Jupyter / Colab: usually the user just wants to
            # load the functions and then call run(...) in a separate cell.
            print("[main] local Jupyter/Colab kernel detected: skipping argparse.")
            print("       Use run(subs_path=..., out_path=..., duration=...).")
    else:
        sys.exit(main() or 0)